# Phase Average Value

Here I test the behavior of the "phase_average_value" function

# Function Implementation

In [ ]:
import numpy as np


def phase_average_value(
    phase_function_values: np.ndarray,
    n_phase_transitions_per_composition: np.ndarray,
    parameter_values: np.ndarray,
    composition_index: int,
    operation: str,
) -> float:
    """
    Compute the averaged material parameter for a composition using
    phase-function-weighted averaging.

    Parameters
    ----------
    phase_function_values : np.ndarray
        Phase function values for all phase transitions.
    n_phase_transitions_per_composition : np.ndarray
        Number of phase transitions associated with each composition.
    parameter_values : np.ndarray
        Material parameter values for each phase.
    composition_index : int
        Index of the composition to evaluate.
    operation : str
        Averaging operation. Must be either "arithmetic" or "logarithmic".

    Returns
    -------
    float
        Averaged parameter value.
    """

    # Calculate the starting parameter index for this composition
    start_phase_index = np.sum(
        n_phase_transitions_per_composition[:composition_index] + 1
    )

    averaged_parameter = parameter_values[start_phase_index]

    if n_phase_transitions_per_composition[composition_index] > 0:

        if operation == "logarithmic":
            averaged_parameter = np.log(averaged_parameter)

        for i in range(n_phase_transitions_per_composition[composition_index]):
            phase_index = start_phase_index + i

            assert(phase_index + 1 < len(parameter_values))

            weight = phase_function_values[phase_index - composition_index]

            if operation == "logarithmic":
                averaged_parameter += weight * (
                    np.log(parameter_values[phase_index + 1]) - averaged_parameter
                )

            elif operation == "arithmetic":
                averaged_parameter += weight * (
                    parameter_values[phase_index + 1] - averaged_parameter
                )

            else:
                raise ValueError(
                    f"Unknown averaging operation '{operation}'. "
                    "Expected 'arithmetic' or 'logarithmic'."
                )

        if operation == "logarithmic":
            averaged_parameter = np.exp(averaged_parameter)

    return averaged_parameter


# Simple Tests

In [ ]:
# This simple tests contain a 2-composition system, 
# with 2 phase transitions for composition 0
# and 1 phase transition for composition 1
n_phase_transitions = np.array([2, 1])
parameter_values = np.array([10.0, 20.0, 30.0, 40.0, 50.0])

# Example 1: assign full transition of phase transition 0 for index 0 
phase_function_values1 = np.array([1.0, 0.0, 0.0])
value1 = phase_average_value(
    phase_function_values1,
    n_phase_transitions,
    parameter_values,
    composition_index=0,
    operation="arithmetic",
)

print("Example 1: assign full transition of phase transition 0 for index 0")
print("\t" + str(value1))
print("\tExpected: ")
print("\t" + str(parameter_values[1]))

# Example 2: assign full transition of phase transition 0 and 1 for index 0 
phase_function_values2 = np.array([1.0, 1.0, 0.0])
value2 = phase_average_value(
    phase_function_values2,
    n_phase_transitions,
    parameter_values,
    composition_index=0,
    operation="arithmetic",
)

print("Example 2: assign full transition of phase transition 0 and 1 for index 0")
print("\t" + str(value2))
print("\tExpected: ")
print("\t" + str(parameter_values[2]))

# Example 3: assign partial transition of phase transition 0 for index 0 
phase_function_values3 = np.array([0.5, 0.0, 0.0])
value3 = phase_average_value(
    phase_function_values3,
    n_phase_transitions,
    parameter_values,
    composition_index=0,
    operation="arithmetic",
)

print("Example 3: assign partial transition of phase transition 0 for index 0")
print("\t" + str(value3))
print("\tExpected: ")
expected3 = parameter_values[0] + 0.5 * (parameter_values[1]-parameter_values[0])
print("\t" + str(expected3))

# Example 4: assign partial transition of phase transition 0 and 1 for index 0 
phase_function_values4 = np.array([0.4, 0.4, 0.0])
value4 = phase_average_value(
    phase_function_values4,
    n_phase_transitions,
    parameter_values,
    composition_index=0,
    operation="arithmetic",
)
print("Example 4: assign partial transition of phase transition 0 and 1 for index 0")
print("\t" + str(value4))
print("\tExpected: ")
expected4 = parameter_values[0] + 0.4 * (parameter_values[1]-parameter_values[0]) + 0.4 * (parameter_values[2]-parameter_values[1])
print("\t" + str(expected4))

# Modified function implementation

In [ ]:
def phase_average_value_modified(
    phase_function_values: np.ndarray,
    n_phase_transitions_per_composition: np.ndarray,
    parameter_values: np.ndarray,
    composition_index: int,
    operation: str,
    *,
    phase_kinetics_values: np.ndarray = np.array([]),
    phase_kinetics_mapping: np.ndarray = np.array([]),
) -> float:
    """
    Compute the averaged material parameter for a composition using
    phase-function-weighted averaging.

    Parameters
    ----------
    phase_function_values : np.ndarray
        Phase function values for all phase transitions.
    n_phase_transitions_per_composition : np.ndarray
        Number of phase transitions associated with each composition.
    parameter_values : np.ndarray
        Material parameter values for each phase.
    phase_kinetics_values: np.ndarray
        Phase transition kinetics values for transition marked with kinetics
    phase_kinetics_mapping: np.ndarray
        Mapping of phase transition kinetics to transitions
    composition_index : int
        Index of the composition to evaluate.
    operation : str
        Averaging operation. Must be either "arithmetic" or "logarithmic".

    Returns
    -------
    float
        Averaged parameter value.
    """

    # Calculate the starting parameter index for this composition
    start_phase_index = np.sum(
        n_phase_transitions_per_composition[:composition_index] + 1
    )

    averaged_parameter = parameter_values[start_phase_index]

    if n_phase_transitions_per_composition[composition_index] > 0:

        if operation == "logarithmic":
            averaged_parameter = np.log(averaged_parameter)

        for i in range(n_phase_transitions_per_composition[composition_index]):
            phase_index = start_phase_index + i

            assert(phase_index + 1 < len(parameter_values))

            weight = phase_function_values[phase_index - composition_index]

            current_parameter = parameter_values[phase_index + 1]

            # Modify the parameter rather than the function itself
            for j in range(phase_kinetics_mapping.size):
                mapping = phase_kinetics_mapping[j]
                start_parameter = parameter_values[start_phase_index+mapping]
                if mapping <= phase_index:
                    current_parameter =  start_parameter + (current_parameter - start_parameter) * phase_kinetics_values[j]

            if operation == "logarithmic":
                averaged_parameter += weight * (
                    np.log(current_parameter) - averaged_parameter
                )

            elif operation == "arithmetic":
                averaged_parameter += weight * (
                    current_parameter - averaged_parameter
                )

            else:
                raise ValueError(
                    f"Unknown averaging operation '{operation}'. "
                    "Expected 'arithmetic' or 'logarithmic'."
                )

        if operation == "logarithmic":
            averaged_parameter = np.exp(averaged_parameter)

    return averaged_parameter


In [ ]:
# This simple tests contain a 2-composition system, 
# with 2 phase transitions for composition 0
# and 1 phase transition for composition 1
n_phase_transitions = np.array([2, 1])
parameter_values = np.array([10.0, 20.0, 30.0, 40.0, 50.0])

# Example 1: assign full transition of phase transition 0 for index 0 
phase_function_values1 = np.array([1.0, 0.0, 0.0])
value1 = phase_average_value_modified(
    phase_function_values1,
    n_phase_transitions,
    parameter_values,
    composition_index=0,
    operation="arithmetic",
)

print("Example 1: assign full transition of phase transition 0 for index 0")
print("\t" + str(value1))
print("\tExpected: ")
print("\t" + str(parameter_values[1]))

# Example 2: assign full transition of phase transition 0 and 1 for index 0 
phase_function_values2 = np.array([1.0, 1.0, 0.0])
value2 = phase_average_value_modified(
    phase_function_values2,
    n_phase_transitions,
    parameter_values,
    composition_index=0,
    operation="arithmetic",
)

print("Example 2: assign full transition of phase transition 0 and 1 for index 0")
print("\t" + str(value2))
print("\tExpected: ")
print("\t" + str(parameter_values[2]))

# Example 3: assign partial transition of phase transition 0 for index 0 
phase_function_values3 = np.array([1.0, 0.0, 0.0])
phase_kinetics_values3 = np.array([0.5,])
phase_kinetics_mapping3 = np.array([0,])
value3 = phase_average_value_modified(
    phase_function_values3,
    n_phase_transitions,
    parameter_values,
    composition_index=0,
    operation="arithmetic",
    phase_kinetics_values=phase_kinetics_values3,
    phase_kinetics_mapping=phase_kinetics_mapping3
)

print("Example 3: assign partial transition of phase transition 0 for index 0")
print("\t" + str(value3))
print("\tExpected: ")
expected3 = parameter_values[0] + 0.5 * (parameter_values[1]-parameter_values[0])
print("\t" + str(expected3))

# Example 4: assign partial transition of phase transition 0 and 1 for index 0 
phase_function_values4 = np.array([1.0, 1.0, 0.0])
phase_kinetics_values4 = np.array([0.4,])
phase_kinetics_mapping4 = np.array([0,])
value4 = phase_average_value_modified(
    phase_function_values4,
    n_phase_transitions,
    parameter_values,
    composition_index=0,
    operation="arithmetic",
    phase_kinetics_values=phase_kinetics_values4,
    phase_kinetics_mapping=phase_kinetics_mapping4
)
print("Example 4: assign partial transition of phase transition 0 and 1 for index 0")
print("\t" + str(value4))
print("\tExpected: ")
expected4 = parameter_values[0] + 0.4 * (parameter_values[1]-parameter_values[0]) + 0.4 * (parameter_values[2]-parameter_values[1])
print("\t" + str(expected4))

# Example 5: assign full transition of phase transition 0, and particle transition of 1 for index 0 
phase_function_values5 = np.array([1.0, 1.0, 0.0])
phase_kinetics_values5 = np.array([0.6])
phase_kinetics_mapping5 = np.array([1])
value5 = phase_average_value_modified(
    phase_function_values5,
    n_phase_transitions,
    parameter_values,
    composition_index=0,
    operation="arithmetic",
    phase_kinetics_values=phase_kinetics_values5,
    phase_kinetics_mapping=phase_kinetics_mapping5
)
print("Example 5: assign full transition of phase transition 0, and particle transition of 1 for index 0")
print("\t" + str(value5))
print("\tExpected: ")
expected5 = parameter_values[1] + 0.6 * (parameter_values[2]-parameter_values[1])
print("\t" + str(expected5))

In [ ]:
# Extract phase kinetics mapping

In [ ]:
def extract_phase_kinetics(
    n_phase_transitions_per_composition: np.ndarray,
    phase_transition_kinetics_mapping: np.ndarray,
    phase_kinetics_values: np.ndarray,
    composition_index: int,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Extract the phase kinetics information for a single composition.

    Parameters
    ----------
    n_phase_transitions_per_composition
        Number of phase transitions for each composition.

    phase_transition_kinetics_mapping
        One entry per phase transition. A non-negative value gives the
        index into phase_kinetics_values. A negative value means that
        transition has no kinetics variable.

    phase_kinetics_values
        Values of the kinetics variables.

    composition_index
        Composition to extract.

    Returns
    -------
    local_phase_kinetics_values
    local_phase_kinetics_mapping
    """

    # first global transition of this composition
    start = np.sum(n_phase_transitions_per_composition[:composition_index])

    n_transitions = n_phase_transitions_per_composition[composition_index]

    values = []
    mapping = []

    for local_transition in range(n_transitions):
        global_transition = start + local_transition

        kinetics_index = phase_transition_kinetics_mapping[global_transition]

        if kinetics_index >= 0:
            values.append(phase_kinetics_values[kinetics_index])
            mapping.append(local_transition)

    return np.asarray(values), np.asarray(mapping, dtype=int)

In [ ]:
# Example 1: 2 composition, 2 phase transitions for composition 0 and 1 phase transition for composition 1
# 1 kinetic variable
# Mapp the first phase transition of composition 0 to the first kinetic variable
n_phase_transitions = np.array([2, 1])
phase_transition_kinetics_mapping = np.array([0, -1, -1])
phase_kinetics_values = np.array([0.5])

# Extract value and mapping for composition 0
values, mapping = extract_phase_kinetics(
    n_phase_transitions,
    phase_transition_kinetics_mapping,
    phase_kinetics_values,
    composition_index=0,
)

print("Example 1: Extract value and mapping for composition 0")
print(values)   # [0.5]
print(mapping)  # [0]

# Extract value and mapping for composition 1
values, mapping = extract_phase_kinetics(
    n_phase_transitions,
    phase_transition_kinetics_mapping,
    phase_kinetics_values,
    composition_index=1,
)

print("Example 1: Extract value and mapping for composition 1")
print(values)   # [0.5]
print(mapping)  # [0]

# Example 2: 2 composition, 3 phase transitions for composition 0 and 2 phase transition for composition 1
# 2 kinetic variable
# Mapp the first phase transition of composition 0 to the first kinetic variable
# Mapp the third phase transition of composition 0 to the second kinetic variable
# Mapp the second phase transition of composition 1 to the first kinetic variable
n_phase_transitions = np.array([3, 2])
phase_transition_kinetics_mapping = np.array([0, -1, 1, -1, 0])
phase_kinetics_values = np.array([0.5, 0.4])

# Extract value and mapping for composition 0
values, mapping = extract_phase_kinetics(
    n_phase_transitions,
    phase_transition_kinetics_mapping,
    phase_kinetics_values,
    composition_index=0,
)

print("Example 2: Extract value and mapping for composition 0")
print(values)   # [0.5, 0.4]
print(mapping)  # [0, 2]

# Extract value and mapping for composition 1
values, mapping = extract_phase_kinetics(
    n_phase_transitions,
    phase_transition_kinetics_mapping,
    phase_kinetics_values,
    composition_index=1,
)

print("Example 2: Extract value and mapping for composition 1")
print(values)   # [0.5]
print(mapping)  # [1]